# Download the selected MTG-Jamendo tracks (320 kbps MP3)

This notebook reads `split_csv.csv`, downloads the official `raw_30s/audio` archives one at a time, extracts only the requested MP3 files to Google Drive, verifies official SHA-256 checksums, and removes each temporary archive.

**Expected permanent Drive use:** about 65.6 GB for the 7,324 tracks, plus a small amount for manifests and logs. Reserve 70-75 GB.

**Important:** the selected paths span all 100 official archive prefixes. Network transfer can approach the complete 508 GB full-quality collection even though only the selected files are retained. A Colab session may not finish all archives; the notebook is resumable. Reconnect, rerun the setup cells, and run the download cell again. Completed prefixes are skipped.

The dataset is intended for non-commercial research/academic use. Individual tracks retain their Creative Commons licenses.

In [ ]:
# Mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Configuration and imports.
from pathlib import Path, PurePosixPath
from collections import defaultdict
import csv
import hashlib
import json
import os
import re
import shutil
import tarfile
import time

import pandas as pd
import requests
from tqdm.auto import tqdm

DRIVE_ROOT = Path('/content/drive/MyDrive/MTG_Jamendo_7324_full_quality')
AUDIO_ROOT = DRIVE_ROOT / 'audio'
CSV_PATH = DRIVE_ROOT / 'split_csv.csv'
STATUS_PATH = DRIVE_ROOT / 'download_status.json'
LOCAL_ARCHIVE_DIR = Path('/content/mtg_jamendo_archives')

MIRROR_ROOT = 'https://cdn.freesound.org/mtg-jamendo/raw_30s/audio'
OFFICIAL_RAW = 'https://raw.githubusercontent.com/MTG/mtg-jamendo-dataset/master'
ARCHIVE_MANIFEST_URL = OFFICIAL_RAW + '/data/download/raw_30s_audio_sha256_tars.txt'
TRACK_MANIFEST_URL = OFFICIAL_RAW + '/data/download/raw_30s_audio_sha256_tracks.txt'
LICENSE_URL = OFFICIAL_RAW + '/audio_licenses.txt'

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
AUDIO_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
print('Drive output:', DRIVE_ROOT)

In [ ]:
# Upload split_csv.csv once if it is not already saved in the Drive project folder.
if not CSV_PATH.exists():
    from google.colab import files
    print('Choose the project data/split_csv.csv file.')
    uploaded = files.upload()
    if 'split_csv.csv' not in uploaded:
        raise FileNotFoundError('The uploaded file must be named split_csv.csv')
    CSV_PATH.write_bytes(uploaded['split_csv.csv'])

df = pd.read_csv(CSV_PATH, dtype={'TRACK_ID': str, 'PATH': str})
required = {'TRACK_ID', 'PATH', 'DURATION'}
missing_columns = required - set(df.columns)
if missing_columns:
    raise ValueError(f'Missing required CSV columns: {sorted(missing_columns)}')

def normalize_relative_path(value):
    value = str(value).replace('\\', '/').lstrip('/')
    parts = PurePosixPath(value).parts
    if len(parts) != 2 or not re.fullmatch(r'\d{2}', parts[0]) or not parts[1].lower().endswith('.mp3'):
        raise ValueError(f'Unexpected MTG-Jamendo path: {value!r}')
    return f'{parts[0]}/{parts[1]}'

df['REL_PATH'] = df['PATH'].map(normalize_relative_path)
if df['REL_PATH'].duplicated().any():
    raise ValueError('split_csv.csv contains duplicate audio paths')

total_seconds = pd.to_numeric(df['DURATION'], errors='raise').sum()
estimated_gb = total_seconds * 320_000 / 8 / 1_000_000_000
prefix_counts = df['REL_PATH'].str[:2].value_counts().sort_index()
print(f'Tracks: {len(df):,}')
print(f'Duration: {total_seconds / 3600:,.2f} hours')
print(f'Estimated MP3 storage: {estimated_gb:,.2f} GB')
print(f'Archive prefixes required: {len(prefix_counts)} / 100')
display(prefix_counts.rename('tracks').to_frame().T)

In [ ]:
# Download and parse the official checksum manifests.
SESSION = requests.Session()
SESSION.headers.update({'User-Agent': 'MTG-Jamendo-research-downloader/1.0'})

def get_text(url, retries=5):
    last_error = None
    for attempt in range(retries):
        try:
            response = SESSION.get(url, timeout=(30, 300))
            response.raise_for_status()
            return response.text
        except Exception as exc:
            last_error = exc
            time.sleep(min(2 ** attempt, 30))
    raise RuntimeError(f'Unable to download {url}: {last_error}')

def parse_checksum_manifest(text):
    result = {}
    for line in text.splitlines():
        line = line.strip()
        if not line:
            continue
        digest, name = line.split(maxsplit=1)
        result[name.strip()] = digest.lower()
    return result

archive_hashes = parse_checksum_manifest(get_text(ARCHIVE_MANIFEST_URL))
raw_track_hashes = parse_checksum_manifest(get_text(TRACK_MANIFEST_URL))

def member_suffix(name):
    parts = PurePosixPath(str(name).replace('\\', '/').lstrip('./')).parts
    if len(parts) >= 2 and re.fullmatch(r'\d{2}', parts[-2]) and parts[-1].lower().endswith('.mp3'):
        return f'{parts[-2]}/{parts[-1]}'
    return None

track_hashes = {}
for stored_name, digest in raw_track_hashes.items():
    key = member_suffix(stored_name)
    if key:
        track_hashes[key] = digest

wanted_paths = set(df['REL_PATH'])
missing_hashes = wanted_paths - set(track_hashes)
if missing_hashes:
    raise RuntimeError(f'{len(missing_hashes)} requested tracks are absent from the official checksum manifest')

license_text = get_text(LICENSE_URL)
(DRIVE_ROOT / 'audio_licenses.txt').write_text(license_text, encoding='utf-8')
print(f'Official archive checksums: {len(archive_hashes)}')
print(f'Official track checksums matched: {len(wanted_paths):,}')

In [ ]:
# Resumable downloader and selective, checksum-verified TAR extraction.
def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

def download_resumable(url, destination, expected_sha256, retries=5):
    destination = Path(destination)
    partial = destination.with_suffix(destination.suffix + '.part')

    if destination.exists():
        if sha256_file(destination) == expected_sha256:
            return destination
        destination.unlink()

    last_error = None
    for attempt in range(retries):
        try:
            existing = partial.stat().st_size if partial.exists() else 0
            headers = {'Range': f'bytes={existing}-'} if existing else {}
            response = SESSION.get(url, headers=headers, stream=True, timeout=(30, 300))
            response.raise_for_status()

            append = existing > 0 and response.status_code == 206
            if existing and not append:
                existing = 0
            mode = 'ab' if append else 'wb'
            remaining = int(response.headers.get('Content-Length', 0))
            total = existing + remaining if remaining else None

            with partial.open(mode) as handle, tqdm(
                total=total, initial=existing, unit='B', unit_scale=True,
                unit_divisor=1024, desc=destination.name
            ) as progress:
                for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                    if chunk:
                        handle.write(chunk)
                        progress.update(len(chunk))

            actual = sha256_file(partial)
            if actual != expected_sha256:
                partial.unlink(missing_ok=True)
                raise ValueError(f'Archive checksum mismatch: expected {expected_sha256}, got {actual}')
            os.replace(partial, destination)
            return destination
        except Exception as exc:
            last_error = exc
            if attempt + 1 < retries:
                print(f'Download attempt {attempt + 1} failed: {exc}. Retrying...')
                time.sleep(min(2 ** attempt * 5, 60))
    raise RuntimeError(f'Failed to download {url}: {last_error}')

def existing_track_is_valid(relative_path):
    output = AUDIO_ROOT / relative_path
    return output.exists() and output.stat().st_size > 16_384

def extract_selected(archive_path, selected_paths):
    selected_paths = set(selected_paths)
    needed = {p for p in selected_paths if not existing_track_is_valid(p)}
    if not needed:
        return []

    extracted = []
    with tarfile.open(archive_path, mode='r|') as archive:
        for member in archive:
            if not member.isfile():
                continue
            key = member_suffix(member.name)
            if key not in needed:
                continue

            source = archive.extractfile(member)
            if source is None:
                raise RuntimeError(f'Unable to read {member.name} from {archive_path.name}')
            destination = AUDIO_ROOT / key
            destination.parent.mkdir(parents=True, exist_ok=True)
            temporary = destination.with_suffix('.mp3.part')
            digest = hashlib.sha256()
            with temporary.open('wb') as output:
                while True:
                    chunk = source.read(4 * 1024 * 1024)
                    if not chunk:
                        break
                    output.write(chunk)
                    digest.update(chunk)

            actual = digest.hexdigest()
            expected = track_hashes[key]
            if actual != expected:
                temporary.unlink(missing_ok=True)
                raise ValueError(f'Track checksum mismatch for {key}: expected {expected}, got {actual}')
            os.replace(temporary, destination)
            extracted.append(key)
            needed.remove(key)
            if not needed:
                break

    if needed:
        raise RuntimeError(f'{len(needed)} selected files were not found in {archive_path.name}: {sorted(needed)[:5]}')
    return extracted

print('Downloader functions ready.')

In [ ]:
# MAIN DOWNLOAD CELL. Safe to stop and rerun.
# To process only selected prefixes in one session, use e.g. ['00', '01', '02'].
PREFIXES_TO_RUN = None  # None means all required prefixes.
DELETE_ARCHIVE_AFTER_EXTRACTION = True

wanted_by_prefix = defaultdict(list)
for relative_path in sorted(wanted_paths):
    wanted_by_prefix[relative_path[:2]].append(relative_path)

prefixes = sorted(wanted_by_prefix)
if PREFIXES_TO_RUN is not None:
    requested_prefixes = {str(p).zfill(2) for p in PREFIXES_TO_RUN}
    prefixes = [p for p in prefixes if p in requested_prefixes]

run_status = {'started_utc': pd.Timestamp.utcnow().isoformat(), 'prefixes': {}}
for index, prefix in enumerate(prefixes, start=1):
    selected = wanted_by_prefix[prefix]
    incomplete = [p for p in selected if not existing_track_is_valid(p)]
    if not incomplete:
        print(f'[{index}/{len(prefixes)}] {prefix}: already complete ({len(selected)} tracks)')
        run_status['prefixes'][prefix] = {'state': 'already_complete', 'tracks': len(selected)}
        STATUS_PATH.write_text(json.dumps(run_status, indent=2), encoding='utf-8')
        continue

    archive_name = f'raw_30s_audio-{prefix}.tar'
    expected_archive_hash = archive_hashes.get(archive_name)
    if expected_archive_hash is None:
        raise KeyError(f'No official checksum found for {archive_name}')
    archive_url = f'{MIRROR_ROOT}/{archive_name}'
    archive_path = LOCAL_ARCHIVE_DIR / archive_name

    print(f'[{index}/{len(prefixes)}] {prefix}: need {len(incomplete)} of {len(selected)} tracks')
    try:
        download_resumable(archive_url, archive_path, expected_archive_hash)
        newly_extracted = extract_selected(archive_path, incomplete)
        remaining = [p for p in selected if not existing_track_is_valid(p)]
        if remaining:
            raise RuntimeError(f'{prefix} still has {len(remaining)} missing tracks')
        run_status['prefixes'][prefix] = {
            'state': 'complete', 'tracks': len(selected), 'newly_extracted': len(newly_extracted)
        }
        print(f'{prefix}: complete; extracted {len(newly_extracted)} new tracks')
    except Exception as exc:
        run_status['prefixes'][prefix] = {'state': 'failed', 'error': repr(exc)}
        STATUS_PATH.write_text(json.dumps(run_status, indent=2), encoding='utf-8')
        raise
    finally:
        if DELETE_ARCHIVE_AFTER_EXTRACTION and archive_path.exists():
            archive_path.unlink()
    STATUS_PATH.write_text(json.dumps(run_status, indent=2), encoding='utf-8')

print('Requested prefix run finished.')

In [ ]:
# Final completeness report. This performs size/existence checks only; extraction already verified SHA-256.
records = []
for row in df.itertuples(index=False):
    relative_path = row.REL_PATH
    audio_path = AUDIO_ROOT / relative_path
    records.append({
        'TRACK_ID': row.TRACK_ID,
        'REL_PATH': relative_path,
        'AUDIO_PATH': str(audio_path),
        'EXISTS': audio_path.exists(),
        'SIZE_BYTES': audio_path.stat().st_size if audio_path.exists() else 0,
        'EXPECTED_SHA256': track_hashes[relative_path],
    })

report = pd.DataFrame(records)
report.to_csv(DRIVE_ROOT / 'selected_audio_manifest.csv', index=False)
complete = report['EXISTS'] & (report['SIZE_BYTES'] > 16_384)
print(f'Complete files: {complete.sum():,} / {len(report):,}')
print(f'Stored audio: {report.loc[complete, "SIZE_BYTES"].sum() / 1_000_000_000:,.2f} GB')
if not complete.all():
    print('Missing/incomplete tracks by archive prefix:')
    display(report.loc[~complete].assign(PREFIX=lambda x: x['REL_PATH'].str[:2])['PREFIX'].value_counts().sort_index())
    print('Rerun the MAIN DOWNLOAD CELL to resume.')
else:
    print('All selected audio files are present and were checksum-verified during extraction.')
display(report.head())